# Speed or promise? Statistical tests of H1-H5 (Olist)

**Input:** `olist.model_data` (built by `05a_export_model_data.sql`), one row per delivered order. Either save it as `model_data.csv` next to this notebook, or let the loader pull it from BigQuery (needs `google-cloud-bigquery` and `gcloud auth application-default login`, or Colab auth).

**Run order:** sections 0-2 first (load, checks, variables), then H1 to H5 in any order.

**Assumptions to confirm before trusting H4**
- `handover_gap_days` = days the carrier handover came after the seller's shipping limit date (positive = seller late). Check the definition in the `02_cleaning.sql` header.
- `gap_days` = delivery date minus promised date in days (positive = late), and `is_late = gap_days > 0`.
- `review_gap_days` = (earlier of answer time and delivery time) minus promised date. Where the review was answered after delivery it should equal `gap_days` (checked in section 1).

Everything here is observational. Coefficients are associations after controls, not causal effects.

In [ ]:
import os, warnings
import numpy as np, pandas as pd, patsy
import statsmodels.formula.api as smf
from scipy.optimize import minimize
from scipy.special import expit
from types import SimpleNamespace
from scipy import stats
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

PROJECT = 'olist-delivery-analytics'   # BigQuery project id
CSV = 'model_data.csv'                 # used if the file exists, otherwise BigQuery
CAP = 30                               # |gap| days are capped at this in regressions (heavy tails)

## 0. Load and type-fix

In [ ]:
if os.path.exists(CSV):
    df = pd.read_csv(CSV)
else:
    from google.cloud import bigquery
    df = bigquery.Client(project=PROJECT).query('SELECT * FROM `olist.model_data`').to_dataframe()

def to_bool(s):
    return s.map({True: True, False: False, 'true': True, 'false': False, 'True': True, 'False': False}).fillna(False).astype(bool)

for c in ['is_late', 'has_timeline_issue', 'answered_before_delivery']:
    df[c] = to_bool(df[c])
for c in ['items_price', 'freight_total', 'distance_km', 'review_score', 'review_gap_days', 'delivery_days',
          'promised_days', 'gap_days', 'handover_days', 'carrier_days', 'handover_gap_days', 'n_items', 'n_sellers']:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype(float)
df['purchase_month'] = pd.to_datetime(df['purchase_month'])
print(df.shape)

## 1. Reconcile to the anchor numbers and check the new columns

In [ ]:
got = {'orders': len(df), 'late_orders': int(df.is_late.sum()), 'no_review': int(df.review_score.isna().sum()),
       'items_revenue': round(float(df.items_price.sum()), 2)}
want = {'orders': 96470, 'late_orders': 6534, 'no_review': 646, 'items_revenue': 13220248.93}
print(got)
bad = {k: (got[k], want[k]) for k in want if abs(got[k] - want[k]) > 0.011}
assert not bad, f'anchor mismatch: {bad}'
assert df.order_id.is_unique

print('late == gap>0 :', (df.is_late == (df.gap_days > 0)).all())
post = df[df.review_score.notna() & ~df.answered_before_delivery]
print('review_gap_days == gap_days for post-delivery reviews:', round((post.review_gap_days == post.gap_days).mean(), 4))
print('missing distance:', df.distance_km.isna().sum(), '| distance km p50/p90/max:',
      df.distance_km.quantile([.5, .9, 1]).round(0).tolist())
print('promised_days > 60:', int((df.promised_days > 60).sum()), '| delivery_days > 60:', int((df.delivery_days > 60).sum()))
print(df.groupby('is_late').review_score.agg(['count', 'mean']).round(3))

## 2. Modelling variables, controls, helpers

In [ ]:
df['gap_cc'] = df.gap_days.clip(-CAP, CAP)
df['late_days'] = df.gap_days.clip(0, CAP)
df['early_days'] = (-df.gap_days).clip(0, CAP)
df['deliv_c'] = df.delivery_days.clip(upper=df.delivery_days.quantile(.99))
df['prom_c'] = df.promised_days.clip(upper=df.promised_days.quantile(.99))
df['log_price'] = np.log1p(df.items_price)
df['log_freight'] = np.log1p(df.freight_total)
df['dist_missing'] = df.distance_km.isna().astype(int)
df['distance_km'] = df.distance_km.fillna(df.groupby('customer_state').distance_km.transform('median')).fillna(df.distance_km.median())
df['log_dist'] = np.log1p(df.distance_km)
df['multi_seller'] = (df.n_sellers > 1).astype(int)
df['month'] = df.purchase_month.dt.strftime('%Y-%m')
df.loc[df.purchase_month < '2017-01-01', 'month'] = '2016'          # pool the 3 tiny 2016 months
top_cat = df.main_category.value_counts().head(25).index
df['cat'] = df.main_category.where(df.main_category.isin(top_cat), 'other').fillna('other')
df['seller'] = df.main_seller_id.fillna('unknown')
df['gbin'] = pd.cut(df.gap_days, [-np.inf, -15, -8, -1, 0, 3, 7, 14, np.inf],
                    labels=['1:15+ early', '2:8-14 early', '3:1-7 early', '4:on day',
                            '5:1-3 late', '6:4-7 late', '7:8-14 late', '8:15+ late']).astype(str)

CTRL = 'C(customer_state) + C(month) + C(cat) + log_price + log_freight + log_dist + dist_missing + multi_seller'

rev = df[df.review_score.notna()].copy()            # modelling sample: reviewed orders
rev_post = rev[~rev.answered_before_delivery]       # reviews answered after delivery
print(len(rev), len(rev_post))

def ols(formula, data, weights=None):
    mod = (smf.ols(formula, data) if weights is None else smf.wls(formula, data, weights=weights))
    if int(mod.nobs) != len(data):
        raise ValueError('rows dropped by patsy (NaN in a used column); clean the frame first')
    return mod.fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(data['seller'])[0]})

def ologit(rhs, data):
    # Ordered logit (scores 1-5) by L-BFGS with analytic gradients. No intercept: the thresholds play that role.
    # statsmodels' OrderedModel gives the same log-likelihood but uses numerical gradients and is far too slow here.
    X = patsy.dmatrix(rhs, data, return_type='dataframe').drop(columns='Intercept')
    Xv = X.values.astype(float); sd = Xv.std(0); Xv = (Xv - Xv.mean(0)) / np.where(sd > 0, sd, 1)
    y = data['review_score'].astype(int).values - 1
    K, k, n = 5, Xv.shape[1], len(y)
    def nll(p):
        c = np.cumsum(np.r_[p[k], np.exp(p[k + 1:])]); eta = Xv @ p[:k]
        cc = np.r_[-np.inf, c, np.inf]
        Fu, Fl = expit(cc[y + 1] - eta), expit(cc[y] - eta)
        pr = np.clip(Fu - Fl, 1e-300, None)
        fu, fl = Fu * (1 - Fu), Fl * (1 - Fl)
        g_b = Xv.T @ (-(fu - fl) / pr)
        up, lo = y < K - 1, y > 0
        g_c = (np.bincount(y[up], weights=(fu / pr)[up], minlength=K - 1)
               - np.bincount(y[lo] - 1, weights=(fl / pr)[lo], minlength=K - 1))
        g_t = np.r_[g_c.sum(), np.exp(p[k + 1:]) * np.array([g_c[j:].sum() for j in range(1, K - 1)])]
        return -np.log(pr).sum(), -np.r_[g_b, g_t]
    cum = np.cumsum(np.bincount(y, minlength=K))[:-1] / n
    c0 = np.log(cum / (1 - cum))
    r = minimize(nll, np.r_[np.zeros(k), c0[0], np.log(np.diff(c0))], jac=True, method='L-BFGS-B',
                 options={'maxiter': 2000, 'maxfun': 5000})
    if not r.success:
        print('WARNING: ordered logit did not converge:', r.message)
    kp = len(r.x)
    return SimpleNamespace(llf=-r.fun, params=r.x, aic=2 * kp + 2 * r.fun, bic=kp * np.log(n) + 2 * r.fun)

def lincom(res, w):
    v = np.zeros(len(res.params)); idx = list(res.params.index)
    for k, c in w.items(): v[idx.index(k)] = c
    est = float(v @ res.params.values); se = float(np.sqrt(v @ res.cov_params().values @ v))
    return est, se, float(2 * stats.norm.sf(abs(est / se)))

def joint_wald(res, names):
    idx = list(res.params.index)
    R = np.zeros((len(names), len(idx)))
    for r, n in enumerate(names): R[r, idx.index(n)] = 1
    w = res.wald_test(R, use_f=False, scalar=True)
    return float(w.statistic), float(w.pvalue)

## H1. Does the gap vs the promise explain review score better than raw delivery days?

Same flexible form for both (natural cubic spline, 5 df), same controls, same sample. M1 = speed only, M2 = gap only, M3 = both. If gap beats speed, M2 fits better than M1 and adding gap to M1 helps more than adding speed to M2.

In [ ]:
print(rev[['delivery_days', 'promised_days', 'gap_days']].corr().round(2))

f_speed = f'cr(deliv_c, df=5, constraints="center")'
f_gap = f'cr(gap_cc, df=5, constraints="center")'
ctl = CTRL
specs = {'M0 controls only': ctl,
         'M1 speed': f'{f_speed} + {ctl}',
         'M2 gap': f'{f_gap} + {ctl}',
         'M3 speed + gap': f'{f_speed} + {f_gap} + {ctl}'}
import time; t0 = time.time()
fit = {}
for k, v in specs.items():
    fit[k] = ologit(v, rev); print(k, 'done', round(time.time() - t0), 's')

h1 = pd.DataFrame({k: {'k_params': len(r.params), 'loglik': r.llf, 'AIC': r.aic, 'BIC': r.bic,
                       'dLL_vs_M0': r.llf - fit['M0 controls only'].llf} for k, r in fit.items()}).T
print(h1.round(1))

def lr(big, small):
    stat = 2 * (fit[big].llf - fit[small].llf); df_ = len(fit[big].params) - len(fit[small].params)
    return round(stat, 1), df_, stats.chi2.sf(stat, df_)
print('\nGap adds to speed  (M3 vs M1): LR=%s df=%s p=%.3g' % lr('M3 speed + gap', 'M1 speed'))
print('Speed adds to gap  (M3 vs M2): LR=%s df=%s p=%.3g' % lr('M3 speed + gap', 'M2 gap'))

In [ ]:
# Same question with OLS and seller-clustered Wald tests (the ordered-logit LR tests above ignore clustering by seller)
m_full = ols(f'review_score ~ {f_speed} + {f_gap} + {CTRL}', rev)
n_speed = [n for n in m_full.params.index if 'deliv_c' in n]
n_gap = [n for n in m_full.params.index if 'gap_cc' in n]
print('gap terms | speed terms in the full model: chi2=%.1f p=%.3g' % joint_wald(m_full, n_gap), '|', 'chi2=%.1f p=%.3g' % joint_wald(m_full, n_speed))
for name, f in [('speed only', f_speed), ('gap only', f_gap), ('both', f'{f_speed} + {f_gap}')]:
    r = ols(f'review_score ~ {f} + {CTRL}', rev)
    print(f'{name:11s} R2={r.rsquared:.4f}  adjR2={r.rsquared_adj:.4f}')

## H2. Does each day late cost more than each day early gains?

Piecewise-linear with a kink at gap = 0, days capped at CAP. `cost_minus_gain > 0` supports H2. The late side saturates (see the bin plot), so read these slopes as averages over a curved shape. Five specifications, because reviews answered before delivery measure lateness differently.

In [ ]:
def piecewise(data, gap_col, label):
    d = data.copy()
    d['e'] = (-d[gap_col]).clip(0, CAP); d['l'] = d[gap_col].clip(0, CAP)
    m = ols(f'review_score ~ e + l + {CTRL}', d)
    p = float(np.squeeze(m.t_test('e + l = 0').pvalue))
    return dict(spec=label, n=int(m.nobs), gain_per_early_day=m.params['e'], se_gain=m.bse['e'],
                cost_per_late_day=-m.params['l'], se_cost=m.bse['l'],
                cost_minus_gain=-(m.params['e'] + m.params['l']), p_symmetric=p)

clean = rev[~rev.has_timeline_issue & (rev.multi_seller == 0)]
h2 = pd.DataFrame([
    piecewise(rev, 'gap_days', '1 all reviews, gap at delivery'),
    piecewise(rev_post, 'gap_days', '2 answered after delivery only'),
    piecewise(rev, 'review_gap_days', '3 all reviews, gap as of review time'),
    piecewise(clean, 'gap_days', '4 all, excl. timeline-issue + multi-seller'),
])
print(h2.round(4).to_string(index=False))
h2.to_csv('results_h2_specs.csv', index=False)

In [ ]:
# Does the late slope differ for reviews answered before delivery?
d = rev.copy(); d['e'] = d.early_days; d['l'] = d.late_days; d['pre'] = d.answered_before_delivery.astype(int)
m = ols(f'review_score ~ e + l + pre + l:pre + {CTRL}', d)
print(m.params[['e', 'l', 'pre', 'l:pre']].round(3).to_frame('coef').join(m.bse[['e', 'l', 'pre', 'l:pre']].round(3).to_frame('se')))

In [ ]:
# Bin plot: where is the cliff, and where does it saturate? (reference = 1-7 days early)
REF = '3:1-7 early'
def bin_coefs(data):
    m = ols(f'review_score ~ C(gbin, Treatment("{REF}")) + {CTRL}', data)
    ci = m.conf_int(); rows = {REF: (0, 0, 0)}
    for n in m.params.index:
        if 'gbin' in n:
            rows[n.split('[T.')[1][:-1]] = (m.params[n], ci.loc[n, 0], ci.loc[n, 1])
    return pd.DataFrame(rows, index=['coef', 'lo', 'hi']).T.sort_index()

fig, ax = plt.subplots(figsize=(9, 4.5))
for data, lab, off in [(rev, 'all reviews', -0.1), (rev_post, 'answered after delivery', 0.1)]:
    b = bin_coefs(data); x = np.arange(len(b)) + off
    ax.errorbar(x, b.coef, yerr=[b.coef - b.lo, b.hi - b.coef], fmt='o-', capsize=3, label=lab)
ax.set_xticks(range(len(b))); ax.set_xticklabels(b.index, rotation=30, ha='right')
ax.axhline(0, color='grey', lw=.5); ax.set_ylabel('score vs 1-7 days early (controls in)'); ax.legend()
plt.tight_layout(); plt.savefig('h2_gap_bins.png', dpi=150); plt.show()
print(rev_post.gbin.value_counts().sort_index())

## H3. Does the same lateness hurt more when the promised time was short?

Promise length is mostly set by route, so it is centred (and cut into terciles) **within customer state**. State fixed effects are in the controls. H3 predicts a positive `late x promise` coefficient: a longer promise cushions the same lateness.

In [ ]:
d = rev.copy()
d['e'] = d.early_days; d['l'] = d.late_days
d['prom_dev'] = d.prom_c - d.groupby('customer_state').prom_c.transform('mean')
d['prom_t'] = d.groupby('customer_state').prom_c.transform(lambda s: pd.qcut(s.rank(method='first'), 3, labels=False)).astype(int)

m = ols(f'review_score ~ e + l*prom_dev + {CTRL}', d)
print('late slope per day:', round(m.params['l'], 4), '| late x promise (per +1 promised day):',
      round(m.params['l:prom_dev'], 4), '| p =', round(m.pvalues['l:prom_dev'], 4))
for dev in (-7, 0, 7):
    est, se, p = lincom(m, {'l': 1, 'l:prom_dev': dev}); print(f'late slope at promise {dev:+d} d vs state mean: {est:.3f} (se {se:.3f})')

m3 = ols(f'review_score ~ e + l*C(prom_t) + {CTRL}', d)
names = [n for n in m3.params.index if n.startswith('l:C(prom_t)')]
print('\nlate slope by within-state promise tercile (0 = shortest):')
print('  T0: %.3f' % m3.params['l'], *['  T%s: %.3f' % (n[-2], m3.params['l'] + m3.params[n]) for n in names])
print('slopes equal across terciles: chi2=%.1f p=%.3g' % joint_wald(m3, names))

late = d[d.is_late].copy()
late['late_bin'] = pd.cut(late.gap_days, [0, 3, 7, 14, np.inf], labels=['1-3', '4-7', '8-14', '15+'])
print('\nmean score, late orders (all reviews) by lateness x promise tercile:')
print(late.pivot_table(index='late_bin', columns='prom_t', values='review_score', aggfunc=['mean', 'count'], observed=True).round(2))

## H4. Seller-stage vs carrier-stage lateness

Late orders only. Total lateness is split into seller-stage days (handover after the shipping limit, **assumed definition**) and a carrier-stage remainder. The test is whether a day of each costs the same. Timeline-issue orders are excluded. Pre-delivery reviews react to the missed promise, so the post-delivery sample is the cleaner read.

In [ ]:
s4 = rev[~rev.has_timeline_issue & rev.is_late].dropna(subset=['handover_gap_days']).copy()
s4['seller_late'] = s4.handover_gap_days.clip(lower=0).clip(upper=CAP)
s4['carrier_resid'] = (s4.gap_days - s4.handover_gap_days.clip(lower=0)).clip(-CAP, CAP)
print('late orders with seller-stage delay > 0: %.1f%%' % (100 * (s4.seller_late > 0).mean()))
print(s4[['seller_late', 'carrier_resid']].describe().round(2))
rows = []
for lab, data in [('all reviews', s4), ('answered after delivery', s4[~s4.answered_before_delivery])]:
    m = ols(f'review_score ~ seller_late + carrier_resid + {CTRL}', data)
    est, se, p = lincom(m, {'seller_late': 1, 'carrier_resid': -1})
    rows.append(dict(sample=lab, n=int(m.nobs), per_seller_day=m.params['seller_late'], se_s=m.bse['seller_late'],
                     per_carrier_day=m.params['carrier_resid'], se_c=m.bse['carrier_resid'], diff=est, p_equal=p))
print(pd.DataFrame(rows).round(4).to_string(index=False))

## H5. Who leaves a review?

(a) Worst-case (Manski) bounds on the late vs on-time score gap if the 646 missing reviews were all 1-star or all 5-star. (b) A response model, then inverse-probability weights, and the H2 piecewise model re-run with them. This only covers non-response. The timing selection (answering before delivery) is handled in H2.

In [ ]:
g = df.groupby('is_late').agg(n=('order_id', 'size'), reviewed=('review_score', 'count'), mean=('review_score', 'mean'))
miss = g.n - g.reviewed
lo = (g['mean'] * g.reviewed + 1 * miss) / g.n; hi = (g['mean'] * g.reviewed + 5 * miss) / g.n
print(pd.DataFrame({'n': g.n, 'missing': miss, 'observed_mean': g['mean'], 'worst_low': lo, 'worst_high': hi}).round(3))
print('late minus on-time: observed %.3f | bounds [%.3f, %.3f]' % (g['mean'][True] - g['mean'][False], lo[True] - hi[False], hi[True] - lo[False]))

In [ ]:
r = df.copy()
r['has_review'] = r.review_score.notna().astype(int); r['late_i'] = r.is_late.astype(int)
top_states = r.customer_state.value_counts().head(8).index
r['state_g'] = r.customer_state.where(r.customer_state.isin(top_states), 'other')
r['quarter'] = r.purchase_month.dt.to_period('Q').astype(str)
resp = smf.logit('has_review ~ late_i + C(state_g) + C(quarter) + log_price + log_freight + log_dist', r).fit(
    disp=False, cov_type='cluster', cov_kwds={'groups': pd.factorize(r.seller)[0]})
print('odds ratio of a review, late vs on-time: %.2f (95%% CI %.2f-%.2f)' % (np.exp(resp.params['late_i']), *np.exp(resp.conf_int().loc['late_i'])))

p_resp = resp.predict(r)
w = (1 / p_resp).loc[rev.index]
d = rev.copy(); d['e'] = (-d.gap_days).clip(0, CAP); d['l'] = d.gap_days.clip(0, CAP)
for lab, wt in [('unweighted', None), ('IPW', w)]:
    m = ols(f'review_score ~ e + l + {CTRL}', d, weights=wt)
    print(f'{lab:10s} gain/early day {m.params["e"]:.4f} | cost/late day {-m.params["l"]:.4f}')

## What to send back
Paste the printed output of sections 1, H1, H2 (spec table and the `pre` interaction), H3, H4 and H5, plus `h2_gap_bins.png`. Note any WARNING lines and the runtime of the H1 ordered logits.